In [1]:
import pandas as pd
import numpy as np


In [2]:
# ==========================================
# CAREFLOW: SYNTHETIC HOSPITAL EVENT LOG
# Generate 5,000 patient visits using Python
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

# 1. Set a random seed so results can be reproduced
np.random.seed(42)
rng = np.random.default_rng(42)

# 2. Define the number of patient visits
TOTAL_PATIENTS = 5000

# 3. Define the simulation period
START_DATE = pd.Timestamp("2026-10-05 00:00:00")
END_DATE = pd.Timestamp("2026-11-05 23:59:59")

# 4. Define available staff/resources for each activity
resources = {
    "Registration": ["Registration_01", "Registration_02"],
    "Triage": ["Nurse_01", "Nurse_02", "Nurse_03"],
    "X-Ray": ["Radiographer_01", "Radiographer_02"],
    "Return to Triage": ["Nurse_01", "Nurse_02", "Nurse_03"],
    "Lab Test": ["LabTech_01", "LabTech_02"],
    "Treatment": ["Nurse_01", "Nurse_02"],
    "Doctor Consultation": ["Doctor_01", "Doctor_02", "Doctor_03"],
    "Admission": ["BedManager_01", "BedManager_02"],
    "Discharge": ["Discharge_01", "Discharge_02"]
}

# 5. Define patient categories
patient_categories = ["Adult", "Child", "Older Adult"]
category_probabilities = [0.70, 0.12, 0.18]

# 6. Store all generated events here
all_events = []

# 7. Generate events for each patient visit
for patient_number in range(1, TOTAL_PATIENTS + 1):

    # Create a unique visit ID
    case_id = f"P{patient_number:05d}"

    # Generate the arrival timestamp
    total_seconds = int(
        (END_DATE - START_DATE).total_seconds()
    )

    arrival_time = START_DATE + pd.Timedelta(
        seconds=int(rng.integers(0, total_seconds + 1))
    )

    # Determine the shift from the arrival hour
    hour = arrival_time.hour

    if 6 <= hour < 14:
        shift = "Morning"
    elif 14 <= hour < 22:
        shift = "Afternoon"
    else:
        shift = "Night"

    # Generate patient category
    patient_category = rng.choice(
        patient_categories,
        p=category_probabilities
    )

    department = "Emergency"

    # Record this patient's chronological events
    events = []
    current_time = arrival_time

    # Event 1: Registration
    events.append(("Registration", current_time))

    # Event 2: Triage
    current_time += pd.Timedelta(
        minutes=int(rng.integers(3, 13))
    )
    events.append(("Triage", current_time))

    # Choose the patient's pathway
    pathway = rng.choice(
        [
            "basic",
            "xray",
            "lab",
            "xray_treatment",
            "lab_treatment",
            "xray_lab"
        ],
        p=[0.22, 0.30, 0.12, 0.18, 0.10, 0.08]
    )

    # Event 3: X-Ray, if required
    if pathway in ["xray", "xray_treatment", "xray_lab"]:

        current_time += pd.Timedelta(
            minutes=int(rng.integers(5, 36))
        )
        events.append(("X-Ray", current_time))

        # Simulate the hidden bottleneck:
        # 40% of X-Ray visits return to triage
        if rng.random() < 0.40:

            current_time += pd.Timedelta(
                minutes=int(rng.integers(5, 26))
            )
            events.append(("Return to Triage", current_time))

            # Patient undergoes triage again
            current_time += pd.Timedelta(
                minutes=int(rng.integers(5, 21))
            )
            events.append(("Triage", current_time))

    # Event 4: Lab test, if required
    if pathway in ["lab", "lab_treatment", "xray_lab"]:

        current_time += pd.Timedelta(
            minutes=int(rng.integers(5, 31))
        )
        events.append(("Lab Test", current_time))

    # Event 5: Treatment, if required
    if pathway in ["xray_treatment", "lab_treatment"]:

        current_time += pd.Timedelta(
            minutes=int(rng.integers(5, 31))
        )
        events.append(("Treatment", current_time))

    # Event 6: Doctor consultation
    current_time += pd.Timedelta(
        minutes=int(rng.integers(5, 46))
    )
    events.append(("Doctor Consultation", current_time))

    # Event 7: Admission or discharge
    if rng.random() < 0.08:

        current_time += pd.Timedelta(
            minutes=int(rng.integers(15, 91))
        )
        events.append(("Admission", current_time))

    else:

        current_time += pd.Timedelta(
            minutes=int(rng.integers(5, 31))
        )
        events.append(("Discharge", current_time))

    # Add every event to the main dataset
    for activity, timestamp in events:

        resource = rng.choice(resources[activity])

        all_events.append({
            "case_id": case_id,
            "activity": activity,
            "timestamp": timestamp,
            "department": department,
            "shift": shift,
            "patient_category": patient_category,
            "resource": resource
        })

# 8. Convert the generated records into a DataFrame
df = pd.DataFrame(all_events)

# 9. Sort events chronologically within each patient visit
df = df.sort_values(
    ["case_id", "timestamp"]
).reset_index(drop=True)

# 10. Save the raw event log as a CSV file
output_file = Path("careflow_raw_events_5000_visits.csv")

df.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

# 11. Validate the generated dataset
print("CARE­FLOW DATASET CREATED SUCCESSFULLY")
print("--------------------------------------")
print("Patient visits:", df["case_id"].nunique())
print("Total event records:", len(df))
print("Total columns:", len(df.columns))
print("Average events per visit:",
      round(df.groupby("case_id").size().mean(), 2))
print("Missing values:", df.isnull().sum().sum())
print("CSV saved to:", output_file.resolve())

print("\nActivity counts:")
print(df["activity"].value_counts())

# 12. Check the simulated return-to-triage rate
xray_cases = df.loc[
    df["activity"] == "X-Ray", "case_id"
].nunique()

return_cases = df.loc[
    df["activity"] == "Return to Triage", "case_id"
].nunique()

if xray_cases > 0:
    return_rate = return_cases / xray_cases * 100
    print(
        "\nReturn-to-triage rate among X-Ray visits:",
        round(return_rate, 2), "%"
    )

print("\nFirst 10 event records:")
print(df.head(10).to_string(index=False))

CARE­FLOW DATASET CREATED SUCCESSFULLY
--------------------------------------
Patient visits: 5000
Total event records: 27962
Total columns: 7
Average events per visit: 5.59
Missing values: 0
CSV saved to: C:\Users\suraj\project\CareFlow-Clinical-Pathway\careflow_raw_events_5000_visits.csv

Activity counts:
activity
Triage                 6141
Registration           5000
Doctor Consultation    5000
Discharge              4601
X-Ray                  2815
Lab Test               1476
Treatment              1389
Return to Triage       1141
Admission               399
Name: count, dtype: int64

Return-to-triage rate among X-Ray visits: 40.53 %

First 10 event records:
case_id            activity           timestamp department     shift patient_category        resource
 P00001        Registration 2026-10-07 20:32:41  Emergency Afternoon            Adult Registration_02
 P00001              Triage 2026-10-07 20:42:41  Emergency Afternoon            Adult        Nurse_03
 P00001            Lab

In [3]:
df

,case_id,activity,timestamp,department,shift,patient_category,resource
0,P00001,Registration,2026-10-07 20:32:41,Emergency,Afternoon,Adult,Registration_02
1,P00001,Triage,2026-10-07 20:42:41,Emergency,Afternoon,Adult,Nurse_03
2,P00001,Lab Test,2026-10-07 20:49:41,Emergency,Afternoon,Adult,LabTech_02
3,P00001,Treatment,2026-10-07 21:12:41,Emergency,Afternoon,Adult,Nurse_02
4,P00001,Doctor Consultation,2026-10-07 21:25:41,Emergency,Afternoon,Adult,Doctor_02
...,...,...,...,...,...,...,...
27957,P04999,Discharge,2026-11-04 03:43:30,Emergency,Night,Adult,Discharge_02
27958,P05000,Registration,2026-10-28 14:40:44,Emergency,Afternoon,Adult,Registration_02
27959,P05000,Triage,2026-10-28 14:46:44,Emergency,Afternoon,Adult,Nurse_03
27960,P05000,Doctor Consultation,2026-10-28 14:56:44,Emergency,Afternoon,Adult,Doctor_02


In [8]:
df["patient_category"].unique()

<ArrowStringArray>
['Adult', 'Child', 'Older Adult']
Length: 3, dtype: str

In [9]:
df.to_csv('careflow_raw_events_5000_visits.csv', index=False)

In [10]:
pwd 

'C:\\Users\\suraj\\project\\CareFlow-Clinical-Pathway'

In [11]:
import sys
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Python location:", sys.executable)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Python location: C:\Users\suraj\ad\python.exe
Pandas version: 3.0.6
NumPy version: 2.5.4


In [12]:

%pip install pm4py


   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 828.9 kB/s eta 0:00:03
   ------- -------------------------------- 0.5/2.7 MB 828.9 kB/s eta 0:00:03
   ----------- ---------------------------- 0.8/2.7 MB 663.4 kB/s eta 0:00:03
   ----------- ---------------------------- 0.8/2.7 MB 663.4 kB/s eta 0:00:03
   --------------- ------------------------ 1.0/2.7 MB 653.9 kB/s eta 0:00:03
   --------------- ------------------------ 1.0/2.7 MB 653.9 kB/s eta 0:00:03
   --------------- ------------------------ 1.0/2.7 MB 653.9 kB/s eta 0:00:03
   ------------------- -------------------- 1.3/2.7 MB 595.9 kB/s eta 0:00:03
   ------------------- --------

In [13]:

import pm4py

print("PM4Py version:", pm4py.__version__)
print("PM4Py installed successfully!")




  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




PM4Py version: 2.7.23.9
PM4Py installed successfully!


In [15]:

import pandas as pd

df = pd.read_csv("careflow_raw_events_5000_visits.csv")

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print(df.head())


Dataset loaded successfully!
Rows: 27962
Columns: ['case_id', 'activity', 'timestamp', 'department', 'shift', 'patient_category', 'resource']
  case_id             activity            timestamp department      shift  \
0  P00001         Registration  2026-10-07 20:32:41  Emergency  Afternoon   
1  P00001               Triage  2026-10-07 20:42:41  Emergency  Afternoon   
2  P00001             Lab Test  2026-10-07 20:49:41  Emergency  Afternoon   
3  P00001            Treatment  2026-10-07 21:12:41  Emergency  Afternoon   
4  P00001  Doctor Consultation  2026-10-07 21:25:41  Emergency  Afternoon   

  patient_category         resource  
0            Adult  Registration_02  
1            Adult         Nurse_03  
2            Adult       LabTech_02  
3            Adult         Nurse_02  
4            Adult        Doctor_02  
